In [ ]:
!pip install pandas ipywidgets pymongo ipython

In [2]:
from pymongo import MongoClient
import pandas as pd
uri = "mongodb://localhost:27017/"
def execute(callable,close=False):
    try:
        client = MongoClient(uri)
        callable(client)

        if close:
            client.close()
    except Exception as e:
        raise Exception("Unable to find the document due to the following error: ", e)

def find(cursor):
    for c in cursor:
        print(c)

def aggregate(cursor):
    for c in cursor:
        print(c)

def show_query_results(cursor, drop_id=False):
    results = list(cursor)
    print(f"Found: {len(results)} documents\n")
    
    if drop_id:
        df = pd.DataFrame(results).drop('_id', axis=1)
    else:
        df = pd.DataFrame(results)
    display(df)  
    

In [2]:
execute(lambda client: show_query_results(client.my_db.users.find({})))

Found: 3 documents



,username,email,age,dob,height
0,Alice,alice@gmail.com,45,1979-01-11,155.45
1,Jane,jane@gmail.com,33,1992-02-22,160.35
2,Bob,bob@gmail.com,31,1994-08-13,165.35


In [1]:
import ipywidgets as widgets

query_box = widgets.Textarea(value='{}', description='Query:')
button = widgets.Button(description='Run')
output = widgets.Output()
skip_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Skip:',
    continuous_update=False
)

limit_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description='Limit:',
    continuous_update=False
)


def run(b):
    with output:
        output.clear_output()
        import json
        query = json.loads(query_box.value)
        execute(lambda client: show_query_results(client.my_db.users.find(query,skip=skip_slider.value,limit=limit_slider.value)))

button.on_click(run)
display(query_box,skip_slider,limit_slider, button, output)

Textarea(value='{}', description='Query:')

IntSlider(value=0, continuous_update=False, description='Skip:')

IntSlider(value=10, continuous_update=False, description='Limit:', min=1)

Button(description='Run', style=ButtonStyle())

Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import json
def get_children(client, is_aggregate=False):
    query_box = widgets.Textarea(value='{}', description='Query:', layout=widgets.Layout(
        height='150px'      # Height
    ))
    skip_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=100,
        step=1,
        description='Skip:',
        continuous_update=False
    )
    limit_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=100,
        step=1,
        description='Limit:',
        continuous_update=False
    )
    
    button = widgets.Button(description='Run', button_style="success")
    output = widgets.Output()

    # Database dropdown
    db_dropdown = widgets.Dropdown(
        options=client.list_database_names(),
        description='Database:'
    )

    # Collection dropdown
    collection_dropdown = widgets.Dropdown(
        description='Collection:'
    )
    
    # Buttons
    refresh_btn = widgets.Button(description='🔄 Refresh Lists', button_style='info')

    # Update collections when database changes
    def update_collections(change):
        db_name = change['new']
        collections = client[db_name].list_collection_names()
        collection_dropdown.options = collections
        if collections:
            collection_dropdown.value = collections[0]

    # Initialize collections
    if db_dropdown.options:
        update_collections({'new': db_dropdown.value})
    db_dropdown.observe(update_collections, 'value')

    # Refresh database list
    def refresh_lists(b):
        db_dropdown.options = client.list_database_names()
        update_collections({'new': db_dropdown.value})

    refresh_btn.on_click(refresh_lists)


    def run(b):
        with output:
            output.clear_output()
            import json
            query = json.loads(query_box.value)
            pipelines = []
            if query:
                pipelines.append(query)
            pipelines.append({'$skip':skip_slider.value})
            if limit_slider.value !=0:
                pipelines.append({'$limit':limit_slider.value})
            if is_aggregate:
                execute(lambda client: show_query_results(client[db_dropdown.value][collection_dropdown.value].aggregate(pipelines)))
            else:
                execute(lambda client: show_query_results(client[db_dropdown.value][collection_dropdown.value].find(query,skip=skip_slider.value,limit=limit_slider.value)))


    button.on_click(run)

    return (db_dropdown,refresh_btn,collection_dropdown,query_box, skip_slider, limit_slider, button, output)

def init_tabs(client):
    tab_names = ["Find", "Aggregate"]
    
    # Create separate widgets for each tab
    find_widgets = get_children(client=client)
    aggregate_widgets = get_children(client=client, is_aggregate=True)
    
    # Create VBox containers for each tab
    find_tab = widgets.VBox(find_widgets)
    aggregate_tab = widgets.VBox(aggregate_widgets)
    
    # Create tab widget
    tab = widgets.Tab()
    tab.children = [find_tab, aggregate_tab]
    
    # Set titles
    for i, name in enumerate(tab_names):
        tab.set_title(i, name)
    
    display(tab)

execute(lambda client:init_tabs(client=client))